# Stg Layer — places (Google Maps Text Search, every read)

Source: Bronze `places_raw` (Lakebase `public.places`, written by the search job and by `auto_search` mode `places`), `place_search_results_raw` and `search_keywords_raw`. Added 2026-10-31; **every read kept since 2026-11-02** (before: full rebuild with the latest read only, so a rating change was lost).

| Silver table | Grain | Load |
| --- | --- | --- |
| `stg_places` | `place_id` × `snapshot_ts` (= `COALESCE(updated_at, created_at)` of the Postgres row) | insert-only, every place (food or not, open or not) with `is_food_place` |
| `stg_place_search_results` | `id` (one place returned for one keyword in one search run; `snapshot_ts` = `fetched_at`) | insert-only; `keyword_normalized` computed here |
| `stg_search_keywords` | `keyword_normalized` × `mode` × `snapshot_ts` | insert-only (view `stg_search_keywords_latest`) |
| view **`stg_places_latest`** | `place_id` | latest read of each place, kept only if it is a **food place that is open**; same columns as the old `stg_places` + `snapshot_ts` |

Columns of a read (computed when it is loaded): `ward` = second-to-last part of `formatted_address` (as geocode); `city` from the address (HCMC → the city label geocode uses); `restaurant_name_normalized` = same rule as `stg_restaurant_mentions`; `is_food_place` = a Google type in `FOOD_TYPES` or ending in `_restaurant` (same rule as `stg/restaurant_geocode`).

Columns of `stg_places_latest` that depend on other tables (recomputed on every read of the view): `district` (approximate ❓: Google returns no district since the 2025 wards, so it is the district of the grid keyword (`stg_search_keywords_latest.district`) that found the place most often, NULL when only non-grid keywords found it) and `search_hits` (distinct keywords that found the place; Place Details reads the most-found Google-only places first).

**Since 2026-11-06 (PR B, datle):** each read also keeps the Place Details fields that Text Search returns, parsed from the raw JSON: `primary_type_display_name`, `opening_periods` (`regularOpeningHours.periods`), `google_ward` / `google_district` / `google_city` / `address_components` (`addressComponents`, a Pro field added to both masks). So a place found by search no longer needs a Place Details call (`stg/restaurant_place_details` only reads video restaurants that search has not read recently). Reads before this date keep NULL in these columns.

`website_uri` / `phone_number` / `google_opening_hours` and the rating / price columns are **Enterprise** fields: NULL for places that only `auto_search` found (Text Search **Pro**) until `stg_restaurant_place_details` reads them. Google rating history over time: `gold.fact_place_rating_snapshot`.

The first run after 2026-11-02 renames the old `stg_places` to `stg_places_bak_20261102` (kept) and rebuilds from Bronze, which holds every places read.

In [ ]:
%run ../gold/_udfs

In [ ]:
%run ./_history

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

PLACES_RAW = f"{BRONZE}.places_raw"
HITS_RAW = f"{BRONZE}.place_search_results_raw"
KEYWORDS_RAW = f"{BRONZE}.search_keywords_raw"
MENTIONS = f"{SILVER}.stg_restaurant_mentions"  # geocode columns (stg/restaurant_geocode)
TARGET = f"{SILVER}.stg_places"
HITS = f"{SILVER}.stg_place_search_results"
KEYWORDS = f"{SILVER}.stg_search_keywords"
LATEST_VIEW = f"{SILVER}.stg_places_latest"
FOOD_TYPES = ["restaurant", "food", "cafe", "coffee_shop", "bakery", "bar", "meal_takeaway", "meal_delivery",
              "food_court", "dessert_shop", "ice_cream_shop", "tea_house", "juice_shop", "snack_bar", "diner",
              "bistro", "deli", "pub"]
print(f"Bronze: {PLACES_RAW}, {HITS_RAW}, {KEYWORDS_RAW} · Silver: {TARGET}, {HITS}, {KEYWORDS} · view {LATEST_VIEW}")

In [ ]:
KEYWORDS_DDL = """
    keyword_normalized STRING, mode STRING, snapshot_ts TIMESTAMP, snapshot_date DATE,
    keyword STRING, origin STRING, dish STRING, cuisine STRING, district STRING, priority DOUBLE,
    active BOOLEAN, cooldown_days INT, last_searched_at TIMESTAMP, next_due_at TIMESTAMP, runs INT,
    last_new_videos INT, total_new_videos INT, created_at TIMESTAMP, updated_at TIMESTAMP, etl_date TIMESTAMP
"""
history_load(KEYWORDS_RAW, KEYWORDS, ["keyword_normalized", "mode"], KEYWORDS_DDL)

HITS_DDL = """
    id BIGINT, snapshot_ts TIMESTAMP, snapshot_date DATE, search_run_id STRING, place_id STRING,
    search_keyword STRING, keyword_normalized STRING, result_rank INT, rating DOUBLE, user_rating_count INT,
    source STRING, fetched_at TIMESTAMP, etl_date TIMESTAMP
"""
history_load(HITS_RAW, HITS, "id", HITS_DDL, ts_expr="fetched_at",
             transform=lambda df: df.withColumn("keyword_normalized", F.expr("normalize_keyword(search_keyword)")))

In [ ]:
hcmc_label = "Hồ Chí Minh"
if spark.catalog.tableExists(MENTIONS) and "geo_city" in spark.table(MENTIONS).columns:
    row = spark.sql(f"SELECT max(geo_city) AS c FROM {MENTIONS} WHERE unaccent_vi(lower(geo_city)) LIKE '%ho chi minh%'").collect()[0]
    hcmc_label = row["c"] or hcmc_label
food = ", ".join(f"'{t}'" for t in FOOD_TYPES)


# addressComponents of the raw Text Search JSON (Pro field, asked since 2026-11-06)
COMPONENTS_TYPE = "ARRAY<STRUCT<longText: STRING, shortText: STRING, types: ARRAY<STRING>>>"
COMPONENTS_SQL = f"from_json(get_json_object(CAST(raw AS STRING), '$.addressComponents'), '{COMPONENTS_TYPE}')"


def place_columns(df):
    """Raw places row -> the columns of one read (no column that depends on other tables)."""
    df.createOrReplaceTempView("places_new")
    return spark.sql(f"""
        WITH parts AS (SELECT *, transform(split(formatted_address, ','), x -> trim(x)) AS addr_parts,
                              {COMPONENTS_SQL if "raw" in df.columns else "CAST(NULL AS " + COMPONENTS_TYPE + ")"} AS comps
                       FROM places_new)
        SELECT
            place_id, snapshot_ts,
            {"ingest_ts," if "ingest_ts" in df.columns else ""}
            display_name                                                AS place_name,
            regexp_replace(lower(trim(unaccent_vi(normalize_keyword(display_name)))), '[^a-z0-9 ]', '') AS restaurant_name_normalized,
            formatted_address,
            CASE WHEN size(addr_parts) >= 3 THEN element_at(addr_parts, -2) END AS ward,
            CASE WHEN unaccent_vi(lower(formatted_address)) LIKE '%ho chi minh%' THEN '{hcmc_label}'
                 ELSE element_at(addr_parts, -1) END                    AS city,
            CAST(latitude AS DOUBLE)                                    AS latitude,
            CAST(longitude AS DOUBLE)                                   AS longitude,
            primary_type,
            CAST(types AS ARRAY<STRING>)                                AS place_types,
            CAST(rating AS DOUBLE)                                      AS rating,
            CAST(user_rating_count AS INT)                              AS rating_count,
            price_level,
            CAST(price_range_start_vnd AS BIGINT)                       AS price_start_vnd,
            CAST(price_range_end_vnd AS BIGINT)                         AS price_end_vnd,
            business_status,
            google_maps_uri,
            website_uri,
            national_phone_number                                       AS phone_number,
            nullif(concat_ws('; ', from_json(get_json_object(CAST(regular_opening_hours AS STRING), '$.weekdayDescriptions'),
                                             'array<string>')), '')  AS google_opening_hours,
            -- 2026-11-06 (PR B): the Place Details fields that Text Search also returns, read from the raw JSON
            {"get_json_object(CAST(raw AS STRING), '$.primaryTypeDisplayName.text')" if "raw" in df.columns else "CAST(NULL AS STRING)"} AS primary_type_display_name,
            {"get_json_object(CAST(raw AS STRING), '$.regularOpeningHours.periods')" if "raw" in df.columns else "CAST(NULL AS STRING)"} AS opening_periods,
            coalesce(try_element_at(filter(comps, c -> array_contains(c.types, 'administrative_area_level_3')), 1).longText, try_element_at(filter(comps, c -> array_contains(c.types, 'sublocality_level_1')), 1).longText, try_element_at(filter(comps, c -> array_contains(c.types, 'sublocality')), 1).longText) AS google_ward,
            coalesce(try_element_at(filter(comps, c -> array_contains(c.types, 'administrative_area_level_2')), 1).longText) AS google_district,
            coalesce(try_element_at(filter(comps, c -> array_contains(c.types, 'administrative_area_level_1')), 1).longText) AS google_city,
            CASE WHEN size(comps) > 0 THEN to_json(comps) END            AS address_components,
            COALESCE(arrays_overlap(CAST(types AS ARRAY<STRING>), array({food}))
                     OR exists(CAST(types AS ARRAY<STRING>), t -> endswith(t, '_restaurant')), false) AS is_food_place,
            source,
            snapshot_ts                                                 AS fetched_at
        FROM parts
    """)


PLACES_DDL = """
    place_id STRING, snapshot_ts TIMESTAMP, snapshot_date DATE, place_name STRING, restaurant_name_normalized STRING,
    formatted_address STRING, ward STRING, city STRING, latitude DOUBLE, longitude DOUBLE, primary_type STRING,
    place_types ARRAY<STRING>, rating DOUBLE, rating_count INT, price_level STRING, price_start_vnd BIGINT,
    price_end_vnd BIGINT, business_status STRING, google_maps_uri STRING, website_uri STRING, phone_number STRING,
    google_opening_hours STRING, is_food_place BOOLEAN, source STRING, fetched_at TIMESTAMP, etl_date TIMESTAMP,
    primary_type_display_name STRING, opening_periods STRING, google_ward STRING, google_district STRING,
    google_city STRING, address_components STRING
"""
# 2026-11-06 (PR B): columns added after the table was created (older reads keep NULL)
PLACES_ADDED = {"primary_type_display_name": "STRING", "opening_periods": "STRING", "google_ward": "STRING",
                "google_district": "STRING", "google_city": "STRING", "address_components": "STRING"}
if spark.catalog.tableExists(TARGET):
    missing = [c for c in PLACES_ADDED if c not in spark.table(TARGET).columns]
    if missing:
        spark.sql(f"ALTER TABLE {TARGET} ADD COLUMNS ({', '.join(f'{c} {PLACES_ADDED[c]}' for c in missing)})")
        print(f"➕ {TARGET}: added columns {missing}")
# Bronze holds every places read, so the old full-rebuild table (other columns) is not copied
history_load(PLACES_RAW, TARGET, "place_id", PLACES_DDL, transform=place_columns, latest_view=False, copy_bak=False)

In [ ]:
# Builtin SQL only (a view cannot call the session UDFs): keyword_normalized was computed when the hits were loaded
spark.sql(f"""
    CREATE OR REPLACE VIEW {LATEST_VIEW}
    COMMENT 'Latest read of each Google Maps place found by Text Search, only food places that are open. district / search_hits recomputed from stg_place_search_results.'
    AS
    WITH latest AS (
        SELECT * FROM {TARGET}
        QUALIFY ROW_NUMBER() OVER (PARTITION BY place_id ORDER BY snapshot_ts DESC) = 1
    ),
    hits AS (
        SELECT h.place_id, k.district, COUNT(*) AS hits, MIN(h.fetched_at) AS first_hit
        FROM {HITS} h
        JOIN (SELECT keyword_normalized, district FROM {KEYWORDS}_latest WHERE mode = 'places') k
          ON k.keyword_normalized = h.keyword_normalized
        WHERE k.district IS NOT NULL
        GROUP BY h.place_id, k.district
    ),
    district_guess AS (
        SELECT place_id, max_by(district, struct(hits, -unix_timestamp(first_hit))) AS district
        FROM hits GROUP BY place_id
    ),
    search_hits AS (
        SELECT place_id, COUNT(DISTINCT keyword_normalized) AS search_hits FROM {HITS} GROUP BY place_id
    )
    SELECT
        l.place_id, l.place_name, l.restaurant_name_normalized, l.formatted_address, l.ward, d.district, l.city,
        l.latitude, l.longitude, l.primary_type, l.place_types, l.rating, l.rating_count, l.price_level,
        l.price_start_vnd, l.price_end_vnd, l.business_status, l.google_maps_uri, l.website_uri, l.phone_number,
        l.google_opening_hours, CAST(COALESCE(h.search_hits, 0) AS INT) AS search_hits, l.source, l.fetched_at,
        l.snapshot_ts, l.etl_date, l.primary_type_display_name, l.opening_periods, l.google_ward, l.google_district,
        l.google_city, l.address_components
    FROM latest l
    LEFT JOIN district_guess d ON d.place_id = l.place_id
    LEFT JOIN search_hits h ON h.place_id = l.place_id
    WHERE l.is_food_place AND (l.business_status IS NULL OR l.business_status = 'OPERATIONAL')
""")

In [ ]:
s = spark.sql(f"SELECT COUNT(*) AS n, COUNT(district) AS with_district, COUNT(rating) AS with_rating FROM {LATEST_VIEW}").collect()[0]
r = spark.sql(f"SELECT COUNT(*) AS reads, COUNT(DISTINCT place_id) AS places FROM {TARGET}").collect()[0]
print(f"✅ {LATEST_VIEW}: {s['n']} food places open (of {r['places']} places, {r['reads']} reads in {TARGET}) · "
      f"{s['with_district']} with a district · {s['with_rating']} with a rating")
display(spark.sql(f"SELECT district, COUNT(*) AS n FROM {LATEST_VIEW} GROUP BY 1 ORDER BY 2 DESC"))
print("\n--- stg_places complete ---")